<a href="https://colab.research.google.com/github/smduarte/spbd-2627/blob/main/docs/labs/lab6/SPBD_Labs_spark3_exercises_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Spark SQL Exercises Solutions


In [31]:
#@title Preparation
!wget -q -O os_maias.txt https://www.dropbox.com/s/n24v0z7y79np319/os_maias.txt?dl=0

##1. Sorted Word Frequency

1.1) Create a [Spark SQL](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html) program that counts the number of occurrences of each word in "Os Maias" novel;

a) No sorting required, words and frequency can appear in any order;

b) Sorted by word, in reverse alphabetical order;

c) Sorted by frequency (the words with higher occurrence first).

Note that the sorting should be performed as a transformation (i.e. it should produce an RDD)...

In [ ]:
#@title Exercise 1.1a - WordFrequency: No Sorting

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

try :
  df = spark.read.text('os_maias.txt').withColumnRenamed('value', 'line')
  df.createOrReplaceTempView("Lines")

  df = spark.sql("""
        SELECT word, count(*) AS frequency FROM
           (SELECT explode(regexp_extract_all(line, r'(?U)[^\\W_]+', 0)) AS word
            FROM Lines
            WHERE length(line) > 0
           )
         GROUP BY word""")

  df.show(5)

except Exception as err:
  print(err)

In [ ]:
#@title Exercise 1.1b - WordFrequency: Sorted by word, alphabetically

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

try :
 df = spark.read.text('os_maias.txt').withColumnRenamed('value', 'line')

 df.createOrReplaceTempView("Lines")

 df = spark.sql("""
        SELECT word, count(*) AS frequency FROM
           (SELECT explode(regexp_extract_all(line, r'(?U)[^\\W_]+', 0)) AS word
            FROM Lines
            WHERE length(line) > 0)
         GROUP BY word
        ORDER BY word""")

 df.show(50)

except Exception as err:
  print(err)

In [ ]:
#@title Exercise 1.1c - WordFrequency: Sorted by Frequency

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

try :
  df = spark.read.text('os_maias.txt').withColumnRenamed('value', 'line')

  df.createOrReplaceTempView("Lines")

  df = spark.sql("""
        SELECT word, count(*) AS frequency FROM
           (SELECT explode(regexp_extract_all(line, r'(?U)[^\\W_]+', 0)) AS word
            FROM Lines
            WHERE length(line) > 0)
         GROUP BY word
        ORDER BY frequency DESC""")

  df.show(5)

except Exception as err:
  print(err)

In [ ]:
#@title Exercise 1.1c - WordFrequency: Sorted by Frequency

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

try :
  df = spark.read.text('os_maias.txt').withColumnRenamed('value', 'line')

  df.createOrReplaceTempView("Lines")

  df = spark.sql("""
        SELECT word, count(*) AS frequency FROM
           (SELECT explode(regexp_extract_all(line, r'(?U)[^\\W_]+', 0)) AS word
            FROM Lines
            WHERE length(line) > 0)
         GROUP BY word
        ORDER BY frequency DESC""")

  df.show(5)

except Exception as err:
  print(err)

1.2) Create a Spark Dataframes program that computes the top 10 most used words in "Os Maias" novel.

In [ ]:
#@title Exercise 1.2 - Top 10 Words by frequency

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

try :
  df = spark.read.text('os_maias.txt').withColumnRenamed('value', 'line')

  df.createOrReplaceTempView("Lines")

  df = spark.sql("""
        SELECT word, count(*) AS frequency FROM
           (SELECT explode(regexp_extract_all(line, r'(?U)[^\\W_]+', 0)) AS word
            FROM Lines
            WHERE length(line) > 0
           )
         GROUP BY word
        ORDER BY frequency DESC
        LIMIT 10""")

  df.show(20)

except Exception as err:
  print(err)

##2. Weblog Analysis

Consider a set of log files captured during a DDOS (*Distributed Denial of Service*) attack, containing information for the web accesses performed during the attack to the server.

The log files contain text lines as shown below, with TAB as the separator:

date |IP_source | status_code | operation | URL | execution time |
-|-|-|-|-|-
timestamp  | string | int | string | string| float |
2016-12-06T08:58:35.318+0000|37.139.9.11|404|GET|/codemove/TTCENCUFMH3C|0.026

In [ ]:
#@title Download the dataset
!wget -q -O weblog.txt https://www.dropbox.com/s/0r8902uj9yum7dg/web.log?dl=0
!head -3 weblog.txt

In [ ]:
#@title Apply an header to the dataset file
!echo 'timestamp ip status method url exec_time' | cat - weblog.txt > weblog.csv
!head -3 weblog.csv

2.1. Count the number of unique IP addresses involved in the DDOS attack.


In [ ]:
#@title Exercise 2.1 - Unique IPs

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

try :
  df = spark.read.csv('weblog.csv', sep = ' ', inferSchema=True, header = True)
  df.printSchema()

  df.createOrReplaceTempView("WebLog")

  df = spark.sql("SELECT count(distinct ip) as TotalIps FROM WebLog")

  df.show()

except Exception as err:
  print(err)

2.2. For each interval of 10 seconds, provide the following information: [number of requests, average execution time, maximum time, minimum time]

In [ ]:
#@title Exercise 2.2 - 10-second interval stats

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

try :
  df = spark.read.csv('weblog.csv', sep = ' ', inferSchema=True, header = True)
  df.printSchema()

  df.createOrReplaceTempView("WebLog")

  df = spark.sql("""
      SELECT WINDOW(timestamp, '10 seconds').start AS interval,
                          count(*) AS requests,
                          avg(exec_time) AS avg_time,
                          max(exec_time) AS max_time,
                          min(exec_time) AS min_time
      FROM WebLog
      GROUP BY interval ORDER BY interval""")

  df.show()

except Exception as err:
  print(err)

2.3. Create an inverted index that, for each interval of 10 seconds, has a list of (unique) IPs executing accesses (to each URL).

In [ ]:
#@title Exercise 2.3 - 10-second inverted index

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

try :
  df = spark.read.csv('weblog.csv', sep = ' ', inferSchema=True, header = True)
  df.printSchema()

  df.createOrReplaceTempView("WebLog")

  df = spark.sql("""
      SELECT WINDOW(timestamp, '10 seconds').start AS interval,
             url,
             collect_set(ip) AS ips
      FROM WebLog
      GROUP BY interval, url ORDER BY interval""")

  df.show()

except Exception as err:
  print(err)